# Embedding：从编号到向量

> 前两节完成了 Tokenizer：文本被切成 token，每个 token 分配了一个整数编号。但编号只是编号——7 和 3 之间没有大小关系，模型无法从编号判断两个词是否相似。
>
> 这一节引入 Embedding：把离散的 token ID 映射到一个连续的向量空间。我们从「为什么需要向量」出发，逐步建立对词向量的直觉，最后组装出训练中实际使用的 Embedding 层。

前两节的 Tokenizer 把文本变成了整数序列。举个例子，句子 `"the cat sat"` 经过 Tokenizer 处理后，被切成 `["the", "cat", "sat"]` 三个 token，再通过词表查到对应的 ID 为 `[5, 1, 3]`。模型看到的输入就是这三个整数。

但这些整数只是编号。token 5 是 cat，token 1 是 the，token 3 是 sat——5 并不比 1 "大"，模型也无法从编号判断 cat 和 dog 是否比 cat 和 algorithm 更相似。编号本身不携带语义信息。

Embedding 的思路是把每个 token ID 映射到一个固定长度的实数向量。同一个向量里的不同维度可以表达不同的特征——有些维度可能捕捉了"是否是动物"，有些可能捕捉了"大小"，有些可能捕捉了"抽象程度"。这些特征不是人工挑选的，而是模型在训练中自己发现的。这样一来，cat 和 dog 在大部分维度上数值接近，而 algorithm 和它们在每个维度上都相差很远——两个向量的距离直接反映了语义的远近。

## 1. 从编号到向量

为了把这个直觉变得更具体，下面用一份手工打分的表来看。假设给每个词在四个维度上打分：

| | 尺寸 | 毛茸茸 | 与人亲近 | 独立性 |
|:---|:---|:---|:---|:---|
| cat | 2 | 8 | 6 | 9 |
| dog | 5 | 9 | 9 | 3 |
| algorithm | 0 | 0 | 0 | 0 |

每一行就是一组数值，可以写成一个向量：

```text
cat       → [2, 8, 6, 9]
dog       → [5, 9, 9, 3]
algorithm → [0, 0, 0, 0]
```

维度越多，描述越细。cat 和 dog 在每个维度上都比较接近，algorithm 和它们完全不一样。Embedding 做的事情本质上是相同的——只不过这些特征不是人工挑选的，而是模型在训练过程中自己学出来的。用多个连续的数值联合描述一个 token，数值本身从数据中学来——这就是 Embedding 的核心思想。

### 从颜色 RGB 到词向量

上面用四个维度描述了 cat、dog 和 algorithm，效果比一个编号好得多。这其实就是 Embedding 的核心思路：用多个数值来描述一个东西，而不是只给它一个编号。

但有一个问题：cat 的「毛茸茸」凭什么打 8 分？「与人亲近」的衡量标准是什么？在实际的 Embedding 中，这些数值不是人定的，而是模型自己学出来的。要理解模型是怎么学的，可以先看一个生活中的例子。

颜色有两种描述方式。一种是为每种颜色起一个名字：钴蓝、胭脂红、珊瑚橙……名字越多，需要的词就越多，而且光看名字无法判断两种颜色有多接近。另一种是用 RGB 三个数值来描述，比如 (201, 23, 30)。三个数字就够了，而且判断相似性很直观——两个颜色的数值越接近，看起来就越像。(201, 23, 30) 和 (180, 20, 40) 都是红色系，(23, 180, 201) 是蓝色系，算一下数值的距离就行。

词的表示也面临同样的问题。给每个词一个编号（ID=5 是 cat，ID=12 是 dog），就像给颜色起名字——编号之间没有大小关系，也无法表达「cat 和 dog 很接近，但和 algorithm 差很远」。但如果每个词都有一个类似 RGB 的向量，词与词之间的相似性就可以直接用数值距离来衡量。

这种用多个数值描述单词的方式，叫做分布式表示（distributed representation）。和 one-hot 不同——one-hot 是一个超长的向量，10000 个词就需要 10000 维，其中只有 1 个位置是 1，其余全是 0。分布式表示只用几百维，每一维都是实数，向量之间的距离直接反映语义的远近。

但向量里的数值怎么来？不能随便填，需要从数据中学。要理解学习的过程，需要先搞清楚一件事：上下文如何决定单词的含义。

### 上下文决定含义

用向量表示单词的研究有很多。如果仔细审视这些研究，会发现几乎所有重要方法都基于一个简单的想法：某个单词的含义由它周围的单词形成。

这个想法的含义很直接。单词本身没有含义，单词的含义由它所在的上下文（语境）形成。含义相似的单词经常出现在相似的语境中。比如：

```text
I drink beer.    We drink wine.
I guzzle beer.   We guzzle wine.
```

drink 的附近常有饮料出现，guzzle 的附近也常有饮料出现——drink 和 guzzle 的上下文相似。基于这个观察，可以推断出 guzzle 和 drink 是近义词（guzzle 意为「大口喝」）。

这里所说的上下文，是指某个关注词周围的单词。上下文的大小（即周围单词的数量）称为窗口大小（window size）。窗口大小为 1，上下文包含左右各 1 个单词；窗口大小为 2，上下文包含左右各 2 个单词，以此类推。

```text
语料: You say goodbye and I say hello.

窗口大小为 2，关注词为 goodbye 时：
  You say goodbye and I say hello.
      ←──── 上下文 ────→

  goodbye 的上下文词 = {You, say, and, I}（左侧 2 个 + 右侧 2 个）
```

本节只处理左右单词数量相同、不考虑句子分隔符的情况。

基于这个观察，最直接的做法是统计每个词的上下文中出现了哪些词、各出现多少次，把结果汇总成一张矩阵——称为共现矩阵（co-occurrence matrix）。矩阵的每一行就是一个词的分布式表示：每一维表示对应单词在该词上下文中出现的次数。

共现矩阵能直观地展示「上下文相似 → 向量相似」这一思想。但它有两个明显的局限：矩阵大小是词表大小的平方（词表 10 万就是 10 万×10 万），且每个维度只是原始计数，无法捕捉更复杂的语义关系。

现代 LLM 不再使用这种基于计数的表示，而是通过训练让 Embedding 层自动学到低维密集向量。接下来我们就来看 Embedding 层的具体实现。

In [1]:
# 共现矩阵演示：用一份迷你语料，手工构建词-上下文矩阵
# 语料就用上面介绍过的 "You say goodbye and I say hello."
corpus = "You say goodbye and I say hello ."
# 为简单起见，按空格分词，并统一小写
tokens = corpus.lower().split()
vocab = sorted(set(tokens))
V = len(vocab)

# token → index 映射
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print(f"语料: {corpus}")
print(f"分词后: {tokens}")
print(f"词表 ({V} 个): {vocab}\n")

# 窗口大小设为 2（左右各 2 个词）
window_size = 2

# 初始化 V×V 的零矩阵，行=关注词，列=上下文词
co_matrix = [[0] * V for _ in range(V)]

# 遍历每个位置作为关注词
for center_pos in range(len(tokens)):
    center_word = tokens[center_pos]
    center_idx = word2idx[center_word]
    # 取左右各 window_size 个词作为上下文
    start = max(0, center_pos - window_size)
    end = min(len(tokens), center_pos + window_size + 1)
    for ctx_pos in range(start, end):
        if ctx_pos == center_pos:
            continue
        ctx_word = tokens[ctx_pos]
        ctx_idx = word2idx[ctx_word]
        co_matrix[center_idx][ctx_idx] += 1

# 打印共现矩阵
header = "       " + "  ".join(f"{w:>6s}" for w in vocab)
print("共现矩阵（行 = 关注词，列 = 上下文词）：")
print(header)
for i, row in enumerate(co_matrix):
    row_str = "  ".join(f"{val:6d}" for val in row)
    print(f"{idx2word[i]:>6s} | {row_str}")

print(f"\n★ 关键观察：")
print(f"  每个词由一行 {V} 维向量描述——这一行就是它的分布式表示。")
print(f"  含义相近的词（比如 say 和 goodbye 都是高频词），")
print(f"  它们在上下文中的分布模式也会趋近。")
print(f"  但问题是——词表 {V} 个词就需要 {V}×{V} = {V*V} 个计数，")
print(f"  而且每个维度只是原始频次，无法捕捉间接的语义关联。")


语料: You say goodbye and I say hello .
分词后: ['you', 'say', 'goodbye', 'and', 'i', 'say', 'hello', '.']
词表 (7 个): ['.', 'and', 'goodbye', 'hello', 'i', 'say', 'you']

共现矩阵（行 = 关注词，列 = 上下文词）：
            .     and  goodbye   hello       i     say     you
     . |      0       0       0       1       0       1       0
   and |      0       0       1       0       1       2       0
goodbye |      0       1       0       0       1       1       1
 hello |      1       0       0       0       1       1       0
     i |      0       1       1       1       0       1       0
   say |      1       2       1       1       1       0       1
   you |      0       0       1       0       0       1       0

★ 关键观察：
  每个词由一行 7 维向量描述——这一行就是它的分布式表示。
  含义相近的词（比如 say 和 goodbye 都是高频词），
  它们在上下文中的分布模式也会趋近。
  但问题是——词表 7 个词就需要 7×7 = 49 个计数，
  而且每个维度只是原始频次，无法捕捉间接的语义关联。


## 2. Embedding 查表

Embedding 的维度（通常写作 `embed_dim` 或 `d_model`）决定了每个 token 被表示成多长的向量。下表整理了主流模型的 Embedding 配置，模型名可点击跳转到对应的 HuggingFace config.json 验证数据：

| 模型 | 词表大小 | Embedding 维度 | Embedding 参数量 |
|:---|:---|:---|:---|
| [GPT-2 small](https://huggingface.co/openai-community/gpt2/blob/main/config.json) | 50,257 | 768 | ~39M |
| [Qwen2.5 0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B/blob/main/config.json) | 151,936 | 896 | ~136M |
| [Gemma 3 1B](https://huggingface.co/google/gemma-3-1b-pt/blob/main/config.json) | 262,144 | 1,152 | ~302M |
| [GPT-2 medium](https://huggingface.co/openai-community/gpt2-medium/blob/main/config.json) | 50,257 | 1,024 | ~51M |
| [Gemma 3 4B](https://huggingface.co/google/gemma-3-4b-pt/blob/main/config.json) | 262,144 | 2,560 | ~671M |
| [LLaMA 2 7B](https://huggingface.co/meta-llama/Llama-2-7b-hf/blob/main/config.json) | 32,000 | 4,096 | ~131M |
| [Mistral 7B](https://huggingface.co/mistralai/Mistral-7B-v0.1/blob/main/config.json) | 32,000 | 4,096 | ~131M |
| [Qwen2.5 7B](https://huggingface.co/Qwen/Qwen2.5-7B/blob/main/config.json) | 152,064 | 3,584 | ~545M |
| [LLaMA 3 8B](https://huggingface.co/meta-llama/Meta-Llama-3-8B/blob/main/config.json) | 128,000 | 4,096 | ~524M |
| [DeepSeek V3](https://huggingface.co/deepseek-ai/DeepSeek-V3/blob/main/config.json) | 129,280 | 7,168 | ~927M |
| [LLaMA 3 70B](https://huggingface.co/meta-llama/Meta-Llama-3-70B/blob/main/config.json) | 128,000 | 8,192 | ~1,049M |
| [Qwen2.5 72B](https://huggingface.co/Qwen/Qwen2.5-72B/blob/main/config.json) | 152,064 | 8,192 | ~1,246M |

从表中可以看出几个规律：

- **词表大小差异很大**。GPT-2 的 5 万 token 词表在当时够用，现代多语言模型（Qwen、Gemma）需要 15 万-26 万 token 才能覆盖更多语言。LLaMA 和 Mistral 保守一些，保持在 3.2 万-12.8 万。
- **维度随模型规模增长**。小模型用几百维，7B 级别用 3K-4K 维，70B+ 用 7K-8K 维。维度越高，向量能容纳的信息越丰富。
- **Embedding 参数量 = vocab_size × d_model**。注意 Qwen2.5 0.5B 虽然模型总参数只有 5 亿，但因为词表大，Embedding 层就有 1.36 亿参数——占模型总参数的四分之一以上。小模型配大词表，Embedding 常常是参数大头。

`nn.Embedding` 本质是一张这样的矩阵：

```text
矩阵形状: [vocab_size, d_model]

第 0 行 → token 0 的向量
第 1 行 → token 1 的向量
第 2 行 → token 2 的向量
...
```

给 Embedding 层一个 token ID，它就取出对应那一行。这些向量一开始是随机的，训练过程中模型会不断调整它们。训练之后，经常出现在相似上下文里的词，向量会靠得更近。

In [ ]:
import torch
import torch.nn as nn

# 模拟一个 mini 词表，Embedding = vocab_size × embed_dim 的矩阵
vocab = ["the", "cat", "sat", "on", "mat", "dog", "log"]
vocab_size = len(vocab)
embed_dim = 4

embedding = nn.Embedding(vocab_size, embed_dim)

print(f"词表大小: {vocab_size}, Embedding 维度: {embed_dim}")
print(f"Embedding 权重形状: {embedding.weight.shape}  ← 就是一个 {vocab_size}×{embed_dim} 矩阵")
print(f"\n前 3 行初始值（随机）:\n{embedding.weight[:3]}")

In [ ]:
import torch

# 查表：给一组 token ID，取出对应的向量
sentence_ids = torch.tensor([0, 1, 2, 3, 0, 4])  # "the cat sat on the mat"
vectors = embedding(sentence_ids)                  # 查表 → [6, 4]

print(f"token IDs: {sentence_ids.tolist()}  →  {[vocab[i] for i in sentence_ids.tolist()]}")
print(f"输出形状: {vectors.shape}  ← [{len(sentence_ids)} 个 token, 每个 {embed_dim} 维]")
print()

# 逐个看
for i, (tid, vec) in enumerate(zip(sentence_ids.tolist(), vectors)):
    print(f"  位置 {i}: '{vocab[tid]}' (ID={tid}) → {vec.tolist()}")

# 关键观察：位置 0 和 4 都是 'the'，向量完全相同
# → 同一个 token 不管出现在哪个位置，查出的向量都一样
print(f"\n关键观察：位置 0 和 4 都是 token 'the'，查出的向量完全相同")
print(f"→ 同一个 token 不管出现在哪个位置，Embedding 查出来的向量都一样")
print(f"→ 模型还无法区分词的顺序——这是下一节位置编码要解决的问题")

**Embedding 是怎么训练出来的**

向量不会凭空拥有语义——它需要从数据中学。业界有两种主要做法。

第一种是**预训练词向量**，以 Word2Vec 和 GloVe 为代表。思路是单独训练 Embedding，然后把它当作固定输入喂给下游模型。

以 Word2Vec 的 Skip-gram 为例：取出一个词（比如"猫"），用它的向量去预测周围窗口内的词（"坐在"、"上"、"垫"）。如果预测错了，就调整向量让预测更准。训练几轮下来，经常出现在相似上下文里的词（猫、狗），向量就自然靠得更近。GloVe 的思路类似，但它不是通过预测来训练，而是直接利用全局的词共现统计。

这种做法的优点是 Embedding 训练一次就可以复用。缺点是 Embedding 在下游任务中是固定的，无法根据具体任务微调。

第二种是**端到端训练**，也是现代 LLM 采用的方式。Embedding 矩阵不再单独训练，而是作为模型的一部分参数，和 Transformer 的其他层一起通过反向传播统一更新：

下面用一段可运行的代码来展示这个过程。用一个迷你 Embedding 矩阵和一层 Linear 模拟简化版 Transformer，跑几步训练，观察矩阵如何变化。

端到端训练不需要单独给 Embedding 设计预训练任务。模型的整体目标（比如"预测下一个 token"）会驱动所有参数的更新，Embedding 矩阵也在其中。每一轮训练，梯度从输出层一路传回 Embedding 层，微调每一行的向量值。经过足够多的训练步数，向量就会自然学出有意义的表示——语义相近的词在向量空间中靠得更近。

后面实现 Mini-GPT 时会看到完整的训练流程。

In [1]:
# 端到端训练演示：Embedding 矩阵如何通过反向传播更新
import torch
import torch.nn as nn

torch.manual_seed(42)

# 用一个迷你词表来演示
vocab_size, d_model = 10, 4
embedding = nn.Embedding(vocab_size, d_model)

# 用一层 Linear 模拟简化版的 "Transformer"
linear = nn.Linear(d_model, vocab_size, bias=False)

print("训练前的 Embedding 矩阵：")
print(embedding.weight.data)
print(f"\n形状: {embedding.weight.shape}  ← [vocab_size={vocab_size}, d_model={d_model}]")

# 构造一组假数据：4 个样本，每个样本 3 个 token
input_ids = torch.randint(0, vocab_size, (4, 3))
targets = torch.randint(0, vocab_size, (4, 3))

optimizer = torch.optim.SGD(
    list(embedding.parameters()) + list(linear.parameters()), lr=0.1
)

print(f"\ninput_ids 形状: {input_ids.shape}  ← [batch=4, seq_len=3]")
print(f"targets 形状:   {targets.shape}")
print(f"\n开始训练...\n")

for step in range(5):
    vectors = embedding(input_ids)           # [4, 3, d_model] — 查表取向量
    logits = linear(vectors)                  # [4, 3, vocab_size] — 投影到词表空间
    loss = nn.functional.cross_entropy(
        logits.view(-1, vocab_size),          # [12, vocab_size]
        targets.view(-1)                      # [12]
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"  step {step+1}: loss = {loss.item():.4f}")

print(f"\n训练后的 Embedding 矩阵：")
print(embedding.weight.data)

# 关键观察：比较训练前后同一行的变化
print(f"\n★ 关键观察：")
print(f"  Embedding 矩阵的每一行都通过梯度更新发生了变化。")
print(f"  这就是端到端训练——Embedding 作为模型参数，")
print(f"  和 Transformer 其他层一起被 loss 驱动着更新。")
print(f"  后面实现 Mini-GPT 时会看到完整的训练流程。")


训练前的 Embedding 矩阵：
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055],
        [ 0.6784, -1.2345, -0.0431, -1.6047],
        [-0.7521,  1.6487, -0.3925, -1.4036],
        [-0.7279, -0.5594, -0.7688,  0.7624],
        [ 1.6423, -0.1596, -0.4974,  0.4396],
        [-0.7581,  1.0783,  0.8008,  1.6806],
        [ 0.0349,  0.3211,  1.5736, -0.8455],
        [ 1.3123,  0.6872, -1.0892, -0.3553],
        [-1.4181,  0.8963,  0.0499,  2.2667],
        [ 1.1790, -0.4345, -1.3864, -1.2862]])

形状: torch.Size([10, 4])  ← [vocab_size=10, d_model=4]

input_ids 形状: torch.Size([4, 3])  ← [batch=4, seq_len=3]
targets 形状:   torch.Size([4, 3])

开始训练...

  step 1: loss = 2.8552
  step 2: loss = 2.7944
  step 3: loss = 2.7370
  step 4: loss = 2.6828
  step 5: loss = 2.6314

训练后的 Embedding 矩阵：
tensor([[ 1.9210,  1.4762,  0.9026, -2.1005],
        [ 0.6454, -1.1871, -0.0511, -1.5549],
        [-0.7521,  1.6487, -0.3925, -1.4036],
        [-0.7256, -0.5677, -0.7669,  0.7602],
        [ 1.6553, -0.1718, -0.5176,  0.4

## 3. 工业界的 Embedding 训练实践

前面的 nn.Embedding 查表把概念讲清楚了。但在真实的大语言模型训练中，还有几个工程决策直接影响参数效率和训练稳定性。下面逐一来看，每个话题都对照了实际开源代码中的做法。

**权重共享（Weight Tying）**

模型有两个地方会用到词表大小的矩阵。开头是 Embedding 层：输入 token ID，查表得到向量。结尾是输出层（lm_head）：输入向量，输出词表大小的 logits。两个矩阵形状相同，都是 [vocab_size, d_model]。

GPT-2 的做法是让这两个矩阵共享同一份权重——lm_head.weight 直接指向 wte.weight。这样词表多大，就省多少参数：vocab_size=50257, d_model=768 时省了约 3900 万参数。

但这个技巧在后续的模型中并没有延续。GPT-3 已经不再 tying，LLaMA ≥8B 和 DeepSeek 也都使用独立的 Embedding 和 lm_head 权重。原因是 tying 虽然省参数，但限制了两个矩阵各自学习不同表示的能力——Embedding 需要处理 token 的身份信息，lm_head 需要生成判别性的 logits，二者对权重的要求并不完全相同。规模越大，拆开越划算。

目前 tying 主要用于参数敏感的场景：非常小的模型（如 LLaMA 3.2 1B/3B、GPT-2 124M）或一些轻量设计。它不是大模型的默认选择。

**Weight decay 的作用范围**

训练时通常会加 weight decay（L2 正则化）来限制权重绝对值，防止过拟合。一个常见误解是 Embedding 权重不应该参与 weight decay——"每一行是语义向量，拉向原点会干扰学习"。但主流开源实现的做法与此不同。

nanoGPT（Karpathy）当前版本的逻辑是 `p.dim() >= 2` 的参与 decay——Embedding 权重是二维矩阵，因此在 decay 组内。HuggingFace Transformers 的 Trainer 也只排除 LayerNorm 参数和所有 bias，Embedding 权重照常参与 decay。

不参与 weight decay 的只有两类：所有 bias（维度 < 2）和 LayerNorm/RMSNorm 的 weight。Embedding 权重不在排除之列。

**初始化标准差**

`nn.Embedding` 默认用 N(0, 1) 初始化——标准差为 1。当 d_model=768 时，向量的模长大致是 √768 ≈ 28，这对后续的 Transformer 层来说偏大。

GPT-2 论文的做法很简单："A simple weight initialization of N(0, 0.02) was sufficient."——所有线性层和 Embedding 层的权重都用标准差 0.02 初始化，不随 d_model 变化。LLaMA 沿用了同样的思路，initializer_range 通常也是 0.02 左右。

之所以可以这么简单，是因为模型中大量使用了 LayerNorm——它本身就能将激活值重新归一化到合适的范围，降低了对初始化精度的要求。0.02 对 d_model=768 和 d_model=4096 都适用，不需要缩放公式。

**混合精度下的 Embedding**

用 FP16/BF16 训练可以加速计算、节省显存。PyTorch 的 AMP（Automatic Mixed Precision）对此有内置处理：

- `autocast` 默认将 Embedding 操作保持在 FP32 执行——它不在 FP16 安全算子列表中，前向计算自动使用 FP32。
- 优化器（配合 GradScaler）自动为所有参数维护 FP32 master copy 用于梯度累积和更新，不需要用户手动为 Embedding 单独创建备份。

如果使用 BF16（A100/H100 支持），动态范围与 FP32 相同，甚至不需要 GradScaler，直接用 `autocast(dtype=torch.bfloat16)` 即可。

下面用代码把这几个实践串起来。

In [ ]:
import torch
import torch.nn as nn

# === 权重共享（Weight Tying）===
# GPT-2 的做法：让输入 Embedding 和输出 lm_head 共享权重。小模型常用，大模型通常不共享。
vocab_size, d_model = 10000, 512

class GPTStyleModel(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model)   # token → vector
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)  # vector → logits
        # ★ 关键：让 lm_head 复用 wte 的权重矩阵——省参数，但限制了各自的学习能力
        self.lm_head.weight = self.wte.weight

    def forward(self, input_ids):
        x = self.wte(input_ids)
        return self.lm_head(x)

model = GPTStyleModel(vocab_size, d_model)

# 验证：两个 weight 指向同一块内存
print("=" * 50)
print("1. 权重共享验证")
print(f"   wte.weight 内存地址:       {model.wte.weight.data_ptr()}")
print(f"   lm_head.weight 内存地址:   {model.lm_head.weight.data_ptr()}")
print(f"   是同一块内存:              {model.wte.weight.data_ptr() == model.lm_head.weight.data_ptr()}")
print(f"   节省参数量:                {vocab_size * d_model:,} → {vocab_size * d_model * 4 / 1e6:.1f} MB (fp32)")

# 反向传播验证：同一块内存上的梯度正确累加
input_ids = torch.randint(0, vocab_size, (2, 16))
logits = model(input_ids)
logits.mean().backward()
print(f"   两个 grad 指向同一内存:    {model.wte.weight.grad.data_ptr() == model.lm_head.weight.grad.data_ptr()}")

# --- 不共享 vs 共享：参数量对比 ---
class NoTieModel(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        return self.lm_head(self.wte(input_ids))

tie_params = sum(p.numel() for p in GPTStyleModel(vocab_size, d_model).parameters())
no_tie_params = sum(p.numel() for p in NoTieModel(vocab_size, d_model).parameters())
print(f"\n2. 参数量对比 (vocab={vocab_size:,}, d_model={d_model}):")
print(f"   共享权重: {tie_params:,}")
print(f"   独立权重: {no_tie_params:,}")
print(f"   节省:     {no_tie_params - tie_params:,} ({100 * (no_tie_params - tie_params) / no_tie_params:.1f}%)")

for name, vs, dm in [("GPT-2 small", 50257, 768), ("GPT-2 medium", 50257, 1024)]:
    saved = vs * dm
    print(f"   {name}: vocab={vs:,}, d_model={dm} → 若共享可节省 {saved:,} 参数 ({saved * 4 / 1e6:.1f} MB)")

1. 权重共享验证
   wte.weight 内存地址:       4419747840
   lm_head.weight 内存地址:   4419747840
   是同一块内存:              True
   若共享可节省:              5,120,000 参数 → 20.5 MB (fp32)
   两个 grad 指向同一内存:    True

2. 参数量对比 (vocab=10,000, d_model=512):
   共享权重: 5,120,000
   独立权重: 10,240,000
   节省:     5,120,000 (50.0%)
   GPT-2 small: vocab=50,257, d_model=768 → 若共享可节省 38,597,376 参数 (154.4 MB)
   GPT-2 medium: vocab=50,257, d_model=1024 → 若共享可节省 51,463,168 参数 (205.9 MB)

In [ ]:
# === 参数分组：哪些参数参与 weight decay ===
# 标准做法（nanoGPT, HuggingFace）：bias 和 LayerNorm 权重不参与 decay
# Embedding 权重是 2D 矩阵，照常参与 weight decay
import torch
import torch.nn as nn
import math

def make_param_groups(model, weight_decay=0.1):
    """按维度分组：< 2 维的（bias, LayerNorm weight）不做 decay"""
    decay_params, no_decay_params = [], []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        # bias 是 1D，LayerNorm weight 也是 1D；Embedding/Layer weight 是 2D+
        if param.dim() < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    
    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ]

model2 = GPTStyleModel(vocab_size, d_model)
param_groups = make_param_groups(model2, weight_decay=0.1)

decay_count = sum(p.numel() for p in param_groups[0]['params'])
no_decay_count = sum(p.numel() for p in param_groups[1]['params'])
print(f"参数分组（weight_decay=0.1）：")
print(f"   参与 decay 的参数:   {decay_count:,}  ← Embedding 在这里（2D 矩阵）")
print(f"   不参与 decay 的参数: {no_decay_count:,}  ← bias + LayerNorm（维度 < 2）")

optimizer = torch.optim.AdamW(param_groups, lr=1e-3)
for i, pg in enumerate(optimizer.param_groups):
    wd = pg['weight_decay']
    count = sum(p.numel() for p in pg['params'])
    print(f"   Group {i}: {count:,} params, weight_decay={wd}")

# === 初始化：GPT-2 的做法 ===
# nn.Embedding 默认 N(0, 1)，而 GPT-2/LLaMA 对所有层统一用 N(0, 0.02)
# LayerNorm 的存在降低了对初始化精度的要求，固定 std 即可
print(f"\n初始化对比：")
torch.manual_seed(42)
default_emb = nn.Embedding(100, 16)
custom_emb = nn.Embedding(100, 16)
nn.init.normal_(custom_emb.weight, mean=0.0, std=0.02)

print(f"   nn.Embedding 默认 N(0, 1):   std = {default_emb.weight.std().item():.4f}")
print(f"   GPT-2 做法 N(0, 0.02):       std = {custom_emb.weight.std().item():.4f}")
print(f"   → 更小的初始值，配合 LayerNorm 一起工作")


3. 参数分组与初始化
参数分组（weight_decay=0.1）：
   参与 decay 的参数:   5,120,000  ← Embedding 在这里（2D 矩阵）
   不参与 decay 的参数: 0  ← bias + LayerNorm（维度 < 2）
   Group 0: 5,120,000 params, weight_decay=0.1
   Group 1: 0 params, weight_decay=0.0

初始化对比：
   nn.Embedding 默认 N(0, 1):   std = 0.9958
   GPT-2 做法 N(0, 0.02):       std = 0.0201
   → 更小的初始值，配合 LayerNorm 一起工作

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# === 模拟一次训练 step（含梯度裁剪）===
# 使用修正后的参数分组：bias 和 norm 不做 decay，Embedding 正常参与
model3 = GPTStyleModel(vocab_size, d_model)
optimizer = torch.optim.AdamW(
    make_param_groups(model3, weight_decay=0.1), lr=1e-3
)

batch = torch.randint(0, vocab_size, (4, 32))   # 随机 batch
targets = torch.randint(0, vocab_size, (4, 32))  # 随机 targets

logits = model3(batch)                            # 前向
loss = F.cross_entropy(
    logits.view(-1, vocab_size), targets.view(-1)
)
loss.backward()                                   # 反向
torch.nn.utils.clip_grad_norm_(model3.parameters(), 1.0)  # 梯度裁剪
optimizer.step()                                  # 更新参数
optimizer.zero_grad()

print(f"模拟训练 step:")
print(f"   loss: {loss.item():.4f}")
print(f"   wte.weight 已更新: {model3.wte.weight.grad is None} → 梯度已清零")
print(f"   → 完整流程：forward → loss → backward → clip → step → zero_grad")


4. 模拟训练 step
   loss: 514.6012
   wte.weight 已更新: True → 梯度已清零
   → 完整流程：forward → loss → backward → clip → step → zero_grad

## 小结

这一节所学的内容：

- Token ID 只是编号，不能直接作为模型的数值输入——编号的大小和语义无关
- 稠密向量用多个实数维度联合描述一个 token，维度固定（d_model），向量之间的距离直接反映语义远近
- 单词的含义由上下文决定——含义相似的词出现在相似的语境中。这是 Embedding 能学到有意义向量的理论根基
- 共现矩阵是基于计数的分布式表示，现代 LLM 改用可训练的低维稠密向量
- nn.Embedding 是一张 [vocab_size, d_model] 的可学习矩阵，查表即可取出 token 对应的向量。端到端训练中它和其他参数一起被 loss 驱动更新
- 工业界实践：weight tying 主要用于小模型，大模型通常独立训练 Embedding 和 lm_head；weight decay 标准做法是只排除 bias 和 LayerNorm；初始化用 N(0, 0.02) 而非 1/√d_model；混合精度由 AMP 自动处理

同一个 token 不管出现在哪个位置，查出的向量都相同——模型还需要知道每个 token 在句子中的位置。下一节引入位置编码来解决这个问题。

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。


**作业 1：Embedding 查表**

Token ID 本身没有语义，Embedding 会把 ID 查成向量。

小提示：`embedding_table[token_ids]` 可以一次取多行。

In [ ]:
# 作业 1：Embedding 查表填空
import torch

embedding_table = torch.tensor([
    [1.0, 0.0],  # token 0
    [0.0, 1.0],  # token 1
    [1.0, 1.0],  # token 2
])
token_ids = torch.tensor([2, 0, 1])

# TODO：把下面三引号里的内容替换成你的代码
vectors = """在这里根据 token_ids 从 embedding_table 里取出对应向量"""

assert not isinstance(vectors, str), "请先替换三引号里的占位内容"
expected = torch.tensor([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
assert torch.equal(vectors, expected), vectors
print("✅ 作业 1 通过：你记住了 Embedding 的核心就是查表")

In [ ]:
import torch
import torch.nn as nn

# 作业 2：训练一个 mini Embedding
# 目标：亲手跑一遍训练循环，观察 Embedding 向量如何变化

torch.manual_seed(42)

vocab_size, embed_dim = 5, 2
embedding = nn.Embedding(vocab_size, embed_dim)
optimizer = torch.optim.SGD(embedding.parameters(), lr=0.5)

# 记录训练前的向量
before = embedding.weight.data.clone()

# 训练目标：让 token 0 和 token 1 的向量靠近
for step in range(20):
    vec0 = embedding(torch.tensor(0))
    vec1 = embedding(torch.tensor(1))
    
    # TODO: 构造 loss，让 vec0 和 vec1 的距离尽量小
    loss = """在这里计算 vec0 和 vec1 之间的距离作为 loss"""
    
    assert not isinstance(loss, str), "请先替换占位内容"
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 计算训练前后的距离变化
d_before = (before[0] - before[1]).pow(2).sum().item()
d_after = (embedding.weight.data[0] - embedding.weight.data[1]).pow(2).sum().item()
print(f"训练前距离: {d_before:.4f}")
print(f"训练后距离: {d_after:.4f}")
assert d_after < d_before, "训练后 token 0 和 1 的距离应该更近"
print("✅ 作业 2 通过：你亲手训练了一个 Embedding，体验了梯度如何更新向量。")

**作业 2：训练一个 mini Embedding**

亲手跑一遍训练循环，观察 Embedding 向量如何变化。

目标：让 token 0 和 token 1 的向量靠得更近。

小提示：两个向量的距离可以用 `(vec0 - vec1).pow(2).sum()` 计算。把它当作 loss 来优化，距离就会越来越小。

## 参考资料

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), 2017 — Transformer 原始论文，Embedding 乘以 √d_model 的惯例来自此文
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — 对原始论文的逐行实现，`Embeddings` 类中可以看到 `self.lut(x) * math.sqrt(self.d_model)` 的写法
- Mikolov et al., [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781), 2013 — Word2Vec，分布式表示的经典工作